## Deep Research

One of the classic cross-business Agentic use cases! This is huge.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">A Deep Research agent is broadly applicable to any business area, and to your own day-to-day activities. You can make use of this yourself!
            </span>
        </td>
    </tr>
</table>

In [1]:
# 1. Setup local Arize Phoenix tracing
import os
from openinference.instrumentation.openai import OpenAIInstrumentor
from opentelemetry import trace as otel_trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter

tracer_provider = TracerProvider()
otel_trace.set_tracer_provider(tracer_provider)
span_exporter = OTLPSpanExporter(endpoint="http://localhost:6006/v1/traces")
span_processor = SimpleSpanProcessor(span_exporter)
tracer_provider.add_span_processor(span_processor)
OpenAIInstrumentor().instrument()

# 2. Standard imports
from agents import Agent, WebSearchTool, trace, Runner, function_tool, OpenAIChatCompletionsModel
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
from IPython.display import display, Markdown
from messenger import send_email, push
from openai import AsyncOpenAI


In [2]:
load_dotenv(override=True)

True

In [3]:
# Constants 

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
google_client = AsyncOpenAI(
    base_url=GEMINI_BASE_URL,
    api_key=os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")
)

# Create the model mapping object for the Agents SDK using gemini-2.5-flash-lite
gemini_model = OpenAIChatCompletionsModel(
    model="gemini-3.5-flash",
    openai_client=google_client
)

MODEL_NAME = gemini_model
USE_EMAIL = True
HOW_MANY_SEARCHES = 20


## Strategy for the Deep Research Agent

We are going to do it the bulletproof way.

We are going to orchestrate with code: separate calls to `Runner.run()` for each step in the process.

We will use Structured Outputs at each point.

## We will build 4 Agents:

1. The Search Agent: searches the web for information
2. The Planner Agent: given a question, comes up with a list of searches that should be made
3. The Writer Agent: writes a robust report
4. The Emailer Agent: crafts and sends an email

And then 4 python functions, 1 to call Runner.run() for each of the 4 agents.


## Agent 1: The Search Agent

### OpenAI Hosted Tools

https://openai.github.io/openai-agents-python/tools/#hosted-tools

A paid, quick approach to carrying out managed functionality on OpenAI's cloud.

Their docs surface these tools, but it's worth keeping in mind that they're costly and lock you in to the OpenAI ecosystem.

OpenAI offers the following hosted tools:

`WebSearchTool` lets an agent search the web.  
`FileSearchTool` allows retrieving information from your OpenAI Vector Stores.  
`CodeInterpreterTool` lets the LLM execute code in a sandboxed environment.  
`HostedMCPTool` exposes a remote MCP server's tools to the model.  
`ImageGenerationTool` generates images from a prompt.  
`ToolSearchTool` lets the model load deferred tools, namespaces, or hosted MCP servers on demand.  

### Important note - API charge of WebSearchTool

This currently costs 1 cent per call for OpenAI WebSearchTool. That can add up to about $1 for the next 2 labs. We'll use free and low cost Search tools with other platforms, so feel free to skip running this if the cost is a concern. Also student Christian W. pointed out that OpenAI can sometimes charge for multiple searches for a single call, so it could sometimes cost more than 1 cent per call.

Costs are in the Tools section here: https://developers.openai.com/api/docs/pricing


In [4]:
# Custom search tool using DuckDuckGo/Google search or standard requests
from agents import function_tool
import requests

@function_tool
def web_search(query: str) -> str:
    """
    Search the web for the given query term and return the text results.
    """
    try:
        # Fetch query from DDG HTML search layout safely
        url = f"https://html.duckduckgo.com/html/?q={requests.utils.quote(query)}"
        headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)'}
        resp = requests.get(url, headers=headers, timeout=10)
        if resp.status_code == 200:
            from bs4 import BeautifulSoup
            soup = BeautifulSoup(resp.text, 'html.parser')
            links = soup.find_all('a', class_='result__snippet')
            snippets = [l.get_text() for l in links[:3]]
            if snippets:
                return '\n\n'.join(snippets)
        return f"Could not retrieve search results for: {query}. Try searching alternate keywords."
    except Exception as e:
        return f"Search error: {e}"

INSTRUCTIONS = """
You are a research assistant. Given a search term, you search the web for that term and 
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 words.
Capture the main points and be succinct. Reply only with the summary.
"""
task = "Most popular AI Agent frameworks in 2026"

settings = ModelSettings(tool_choice="required")
tools = [web_search]


In [5]:
search_agent = Agent(name="Search Agent", instructions=INSTRUCTIONS, tools=tools, model=gemini_model, model_settings=settings)

In [6]:
result = await Runner.run(search_agent, task)
display(Markdown(result.final_output))

OPENAI_API_KEY is not set, skipping trace export


In 2026, several AI agent frameworks are anticipated to be prominent players in the field, fundamentally reshaping work and business operations. Among the most frequently cited frameworks are LangGraph, CrewAI, AutoGen, and LlamaIndex. These frameworks are expected to be key in driving advancements in AI agents, impacting areas like decision-making and various business processes.

Other notable frameworks that are predicted to gain traction include Claude SDK, OpenAI Swarm, and Semantic Kernel. Additionally, Haystack is another framework mentioned in the context of time series analysis alongside the other top contenders. The landscape of AI agent frameworks in 2026 will likely involve a strong competition between these tools, with a focus on their capabilities for real-world production, pricing models, and overall effectiveness in building sophisticated AI agents.

OPENAI_API_KEY is not set, skipping trace export


### As always, take a look at the trace

https://platform.openai.com/traces

## Agent 2: The Planner Agent

### We will now use Structured Outputs, and include a description of the fields

In [6]:
class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")
    query: str = Field(description="The search term to use for the web search.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")

In [7]:
WebSearchPlan.model_json_schema()

{'$defs': {'WebSearchItem': {'properties': {'reason': {'description': 'Your reasoning for why this search is important to the query.',
     'title': 'Reason',
     'type': 'string'},
    'query': {'description': 'The search term to use for the web search.',
     'title': 'Query',
     'type': 'string'}},
   'required': ['reason', 'query'],
   'title': 'WebSearchItem',
   'type': 'object'}},
 'properties': {'searches': {'description': 'A list of web searches to perform to best answer the query.',
   'items': {'$ref': '#/$defs/WebSearchItem'},
   'title': 'Searches',
   'type': 'array'}},
 'required': ['searches'],
 'title': 'WebSearchPlan',
 'type': 'object'}

In [8]:
# See note above about cost of WebSearchTool

INSTRUCTIONS = f"""
You are a research assistant. Given a user query, come up with a set of web searches
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for.
"""

planner_agent = Agent(name="Planner Agent", instructions=INSTRUCTIONS, model=gemini_model, output_type=WebSearchPlan)

In [10]:

result = await Runner.run(planner_agent, task)
result.final_output

OPENAI_API_KEY is not set, skipping trace export


WebSearchPlan(searches=[WebSearchItem(reason='To understand the current leading AI agent frameworks, which will likely influence 2026 trends.', query='most popular AI agent frameworks 2024'), WebSearchItem(reason='To identify emerging frameworks and technologies that are gaining traction now and could be dominant in 2026.', query='emerging AI agent frameworks future trends'), WebSearchItem(reason='To find predictions from experts, analysts, or industry publications about future AI agent technology.', query='AI agent framework predictions 2026'), WebSearchItem(reason='To understand the core components and functionalities that define an AI agent framework and make it effective.', query='what is an AI agent framework components'), WebSearchItem(reason='To look for comparisons and reviews of current frameworks, highlighting strengths and weaknesses that might determine future adoption.', query='comparison of AI agent frameworks'), WebSearchItem(reason='To identify key trends in AI that wil

OPENAI_API_KEY is not set, skipping trace export


## Agent 3: The Writer Agent

In [9]:
INSTRUCTIONS = """
You are a senior researcher tasked with writing a cohesive report for a research query.
You will be provided with the original query, and some research.
Generate a comprehensive report based on the research and the query.
The final output should be in markdown format, and it should be lengthy and detailed. Aim 
for 5-10 pages of content, at least 1000 words.
"""


class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")
    markdown_report: str = Field(description="The final report")
    follow_up_questions: list[str] = Field(description="Suggested topics to research further")


writer_agent = Agent(name="Writer Agent", instructions=INSTRUCTIONS, model=gemini_model, output_type=ReportData)

## Agent 4: The email agent

In [10]:
@function_tool
def send_email_tool(subject: str, text_body: str, html_body: str) -> str:
    """
    Send out an email with the given subject and body to all sales prospects
    
    Args:
        subject: The subject of the email
        text_body: The body of the email as plain text
        html_body: The HTML body of the email
    """
    if USE_EMAIL:
        send_email(subject, text_body, html_body)
    else:
        push(f"Subject: {subject}\n\n{text_body}")
    return "Email sent successfully"

In [11]:
send_email_tool.params_json_schema

{'properties': {'subject': {'description': 'The subject of the email',
   'title': 'Subject',
   'type': 'string'},
  'text_body': {'description': 'The body of the email as plain text',
   'title': 'Text Body',
   'type': 'string'},
  'html_body': {'description': 'The HTML body of the email',
   'title': 'Html Body',
   'type': 'string'}},
 'required': ['subject', 'text_body', 'html_body'],
 'title': 'send_email_tool_args',
 'type': 'object',
 'additionalProperties': False}

In [12]:
INSTRUCTIONS = """
You are provided with a detailed report. Use your tool to send an email, converting the report into
a clean, well presented HTML email with an appropriate subject line.
"""

email_agent = Agent(name="Email Agent", instructions=INSTRUCTIONS, tools=[send_email_tool], model=gemini_model)

## Now to Orchestrate by Code

The next 2 functions will plan and execute the search, using the Agents, with calls to `Runner.run()`

In [13]:
async def run_searches(query: str):
    print("Planning searches...")
    result = await Runner.run(planner_agent, f"Query: {query}")
    searches = result.final_output.searches
    print(f"Will perform {len(searches)} searches")
    tasks = [search(item) for item in searches]
    results = await asyncio.gather(*tasks)
    print("Finished searching")
    return results


async def search(item: WebSearchItem):
    input_message = f"Search term: {item.query}\nReason for searching: {item.reason}"
    result = await Runner.run(search_agent, input_message)
    return result.final_output

The next 2 functions write a report and email it

In [14]:
async def write_report(query: str, search_results: list[str]):
    print("Thinking about report...")
    input_message = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await Runner.run(writer_agent, input_message)
    print("Finished writing report")
    return result.final_output

async def send_report_email(report: ReportData):
    print("Writing email...")
    result = await Runner.run(email_agent, report.markdown_report)
    print("Email sent")
    return result.final_output

### Showtime!

In [15]:
import os
import requests
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")

# Query Google's official Gemini models metadata endpoint
url = f"https://generativelanguage.googleapis.com/v1beta/models?key={api_key}"
response = requests.get(url)

if response.status_code == 200:
    models = response.json().get("models", [])
    print("Available Gemini Models:")
    for model in models:
        # Filter for models that support text generation (generateContent)
        if "generateContent" in model.get("supportedGenerationMethods", []):
            model_name = model.get('name').replace('models/', '')
            print(f"- {model_name} ({model.get('displayName')})")
else:
    print(f"Failed to fetch models. Status code: {response.status_code}")
    print(response.text)


Available Gemini Models:
- gemini-2.5-flash (Gemini 2.5 Flash)
- gemini-2.5-pro (Gemini 2.5 Pro)
- gemini-2.0-flash (Gemini 2.0 Flash)
- gemini-2.0-flash-001 (Gemini 2.0 Flash 001)
- gemini-2.0-flash-lite-001 (Gemini 2.0 Flash-Lite 001)
- gemini-2.0-flash-lite (Gemini 2.0 Flash-Lite)
- gemini-2.5-flash-preview-tts (Gemini 2.5 Flash Preview TTS)
- gemini-2.5-pro-preview-tts (Gemini 2.5 Pro Preview TTS)
- gemma-4-26b-a4b-it (Gemma 4 26B A4B IT)
- gemma-4-31b-it (Gemma 4 31B IT)
- gemini-flash-latest (Gemini Flash Latest)
- gemini-flash-lite-latest (Gemini Flash-Lite Latest)
- gemini-pro-latest (Gemini Pro Latest)
- gemini-2.5-flash-lite (Gemini 2.5 Flash-Lite)
- gemini-2.5-flash-image (Nano Banana)
- gemini-3-pro-preview (Gemini 3 Pro Preview)
- gemini-3-flash-preview (Gemini 3 Flash Preview)
- gemini-3.1-pro-preview (Gemini 3.1 Pro Preview)
- gemini-3.1-pro-preview-customtools (Gemini 3.1 Pro Preview Custom Tools)
- gemini-3.1-flash-lite-preview (Gemini 3.1 Flash Lite Preview)
- gemini-

In [ ]:
query ="Most popular AI Agent frameworks in 2026"

with trace("Research trace"):
    print("Starting research...")
    search_results = await run_searches(query)
    report = await write_report(query, search_results)
    await send_report_email(report)  
    print("Hooray!")

Starting research...
Planning searches...


OPENAI_API_KEY is not set, skipping trace export


Will perform 20 searches


OPENAI_API_KEY is not set, skipping trace export


RateLimitError: Error code: 429 - [{'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-3.5-flash\nPlease retry in 42.05273918s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3.5-flash'}, 

OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export


### As always, take a look at the trace

https://platform.openai.com/traces

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thanks.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00cc00;">Congratulations on your progress, and a request</h2>
            <span style="color:#00cc00;">You've reached an important moment with the course; you've created a valuable Agent using one of the latest Agent frameworks. You've upskilled, and unlocked new commercial possibilities. Take a moment to celebrate your success!<br/><br/>Something I should ask you -- my editor would smack me if I didn't mention this. If you're able to rate the course on Udemy, I'd be seriously grateful: it's the most important way that Udemy decides whether to show the course to others and it makes a massive difference.<br/><br/>And another reminder to <a href="https://www.linkedin.com/in/eddonner/">connect with me on LinkedIn</a> if you wish! If you wanted to post about your progress on the course, please tag me and I'll weigh in to increase your exposure.
            </span>
        </td>
    </tr>